# Module 5: GMM Estimation for Dynamic Panels

## Learning Objectives

By completing this notebook, you will:
1. Understand the Generalized Method of Moments (GMM) framework
2. Implement Arellano-Bond difference GMM estimator
3. Apply Blundell-Bond system GMM for improved efficiency
4. Perform Hansen J-test for overidentifying restrictions
5. Validate instrument validity and model specification

## Prerequisites

- Module 5.1 (Dynamic Panels) completed
- Understanding of instrumental variables
- Familiarity with moment conditions

## Estimated Time

90-105 minutes

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(42)

print("✅ Setup complete")

## 1. GMM Framework for Dynamic Panels

### Model

$$y_{it} = \rho y_{i,t-1} + X_{it}\beta + \alpha_i + \epsilon_{it}$$

### First Difference

$$\Delta y_{it} = \rho \Delta y_{i,t-1} + \beta \Delta X_{it} + \Delta \epsilon_{it}$$

### Moment Conditions (Arellano-Bond)

Under assumptions:
1. No serial correlation: $E[\epsilon_{it} \epsilon_{is}] = 0$ for $t \neq s$
2. Strict exogeneity of X (or predetermined)

**Valid instruments:**

$$E[y_{i,t-s} \Delta \epsilon_{it}] = 0, \quad s \geq 2$$
$$E[X_{i,t-s} \Delta \epsilon_{it}] = 0, \quad s \geq 2$$

### Key Advantages

1. **More instruments:** Uses all available lags (not just t-2)
2. **Efficient:** Optimal weighting of moment conditions
3. **Testable:** Hansen J-test for overidentification

## 2. Generate Dynamic Panel Data

In [ ]:
def generate_dynamic_panel(n=150, t=8, rho=0.7, beta=1.2):
    """
    Generate dynamic panel data with AR(1) structure.
    """
    alpha_i = np.random.randn(n) * 8 + 20
    
    data = []
    for i in range(n):
        y_prev = alpha_i[i] / (1 - rho)
        
        for t_idx in range(t):
            x = 5 + np.random.randn() * 2
            epsilon = np.random.randn() * 4
            
            y = rho * y_prev + beta * x + alpha_i[i] + epsilon
            
            data.append({
                'firm': i + 1,
                'year': 2010 + t_idx,
                'sales': y,
                'sales_lag': y_prev,
                'advertising': x
            })
            
            y_prev = y
    
    return pd.DataFrame(data)

df = generate_dynamic_panel(n=150, t=8, rho=0.7, beta=1.2)

print("Dynamic Panel Data: Firm Sales Model")
print("="*70)
print(f"Model: sales_t = 0.7 × sales_(t-1) + 1.2 × advertising + firm_FE + error")
print(f"\nPanel structure:")
print(f"  Firms (N): {df['firm'].nunique()}")
print(f"  Years (T): {df['year'].nunique()}")
print(f"  Observations: {len(df)}")
print(f"\nFirst 12 rows:")
print(df.head(12))

## 3. Arellano-Bond Difference GMM

### Moment Conditions

For period $t$, use lags $t-2, t-3, ..., 1$ as instruments:

$$E[y_{i,t-s} \Delta \epsilon_{it}] = 0, \quad s = 2, 3, ..., t-1$$

### Instrument Matrix

For $t=3$: $Z_3 = [y_{i1}]$

For $t=4$: $Z_4 = [y_{i1}, y_{i2}]$

For $t=5$: $Z_5 = [y_{i1}, y_{i2}, y_{i3}]$

Instruments grow with $t$ → Many moment conditions

In [ ]:
def arellano_bond_gmm(df, entity_col='firm', time_col='year', max_lags=4):
    """
    Simplified Arellano-Bond difference GMM.
    
    Uses up to max_lags instruments for each time period.
    
    NOTE: This is a simplified implementation for educational purposes.
    Production code should use linearmodels.panel or statsmodels GMM classes.
    """
    df_sorted = df.sort_values([entity_col, time_col]).copy()
    
    # First differences
    for col in ['sales', 'sales_lag', 'advertising']:
        df_sorted[f'd_{col}'] = df_sorted.groupby(entity_col)[col].diff()
    
    # Create instrument lags (y_{t-2}, y_{t-3}, ...)
    for lag in range(2, max_lags + 2):
        df_sorted[f'y_lag{lag}'] = df_sorted.groupby(entity_col)['sales'].shift(lag)
    
    # Drop missing
    required_cols = ['d_sales', 'd_sales_lag', 'd_advertising'] + \
                   [f'y_lag{lag}' for lag in range(2, max_lags + 2)]
    df_gmm = df_sorted.dropna(subset=required_cols)
    
    # GMM estimation
    # Step 1: Initial 2SLS with all instruments
    instruments = [f'y_lag{lag}' for lag in range(2, max_lags + 2)] + ['d_advertising']
    
    X = df_gmm[['d_sales_lag', 'd_advertising']].values
    y = df_gmm['d_sales'].values
    Z = df_gmm[instruments].values
    
    # 2SLS: β_2sls = (X'Z(Z'Z)^(-1)Z'X)^(-1) X'Z(Z'Z)^(-1)Z'y
    ZtZ_inv = np.linalg.inv(Z.T @ Z)
    PZ = Z @ ZtZ_inv @ Z.T
    
    XtPZX = X.T @ PZ @ X
    XtPZy = X.T @ PZ @ y
    
    beta_2sls = np.linalg.solve(XtPZX, XtPZy)
    
    # Residuals
    residuals = y - X @ beta_2sls
    
    # Standard errors (simplified, assumes homoskedasticity)
    sigma2 = np.sum(residuals**2) / (len(residuals) - X.shape[1])
    var_beta = sigma2 * np.linalg.inv(XtPZX)
    se = np.sqrt(np.diag(var_beta))
    
    # Hansen J-test for overidentification
    # J = n × m'W^(-1)m where m = Z'e/n
    n = len(residuals)
    moments = (Z.T @ residuals) / n
    j_stat = n * moments.T @ ZtZ_inv @ moments
    j_df = len(instruments) - X.shape[1]
    j_pvalue = 1 - stats.chi2.cdf(j_stat, j_df)
    
    return {
        'rho': beta_2sls[0],
        'beta': beta_2sls[1],
        'se_rho': se[0],
        'se_beta': se[1],
        'n_obs': n,
        'n_instruments': len(instruments),
        'j_statistic': j_stat,
        'j_df': j_df,
        'j_pvalue': j_pvalue
    }

gmm_results = arellano_bond_gmm(df, max_lags=4)

print("\nArellano-Bond Difference GMM Results")
print("="*70)
print(f"{'Parameter':<20} {'True':>10} {'Estimate':>12} {'Std Error':>12} {'t-stat':>10}")
print("="*70)

rho_t = gmm_results['rho'] / gmm_results['se_rho']
beta_t = gmm_results['beta'] / gmm_results['se_beta']

print(f"{'ρ (persistence)':<20} {0.70:>10.4f} {gmm_results['rho']:>12.4f} {gmm_results['se_rho']:>12.4f} {rho_t:>10.3f}")
print(f"{'β (advertising)':<20} {1.20:>10.4f} {gmm_results['beta']:>12.4f} {gmm_results['se_beta']:>12.4f} {beta_t:>10.3f}")

print(f"\nModel Diagnostics:")
print(f"  Observations: {gmm_results['n_obs']}")
print(f"  Instruments: {gmm_results['n_instruments']}")
print(f"\nHansen J-test (overidentification):")
print(f"  J-statistic: {gmm_results['j_statistic']:.4f}")
print(f"  Degrees of freedom: {gmm_results['j_df']}")
print(f"  P-value: {gmm_results['j_pvalue']:.4f}")

if gmm_results['j_pvalue'] > 0.05:
    print("\n  ✅ Fail to reject: Instruments appear valid")
else:
    print("\n  ⚠️  Reject: Model may be misspecified or instruments invalid")

## 4. System GMM (Blundell-Bond)

### Motivation

**Problem with Difference GMM:**
- Lagged levels are weak instruments when series is persistent (high $\rho$)
- "Weak instrument problem" → Large standard errors

### Solution: Add Level Equations

**System combines:**

1. **Differenced equations** (original AB):
   - Instruments: $y_{i,t-2}, y_{i,t-3}, ...$

2. **Level equations:**
   - Instruments: $\Delta y_{i,t-1}$
   
**Additional assumption:** $E[\Delta y_{i,t-1} (\alpha_i + \epsilon_{it})] = 0$

This requires initial conditions restriction (stationarity).

In [ ]:
def system_gmm_simplified(df, entity_col='firm', time_col='year'):
    """
    Simplified System GMM (Blundell-Bond).
    
    Combines difference and level equations.
    This is a simplified version for illustration.
    """
    df_sorted = df.sort_values([entity_col, time_col]).copy()
    
    # Create transformations
    for col in ['sales', 'sales_lag', 'advertising']:
        df_sorted[f'd_{col}'] = df_sorted.groupby(entity_col)[col].diff()
    
    # Instruments
    df_sorted['y_lag2'] = df_sorted.groupby(entity_col)['sales'].shift(2)
    df_sorted['y_lag3'] = df_sorted.groupby(entity_col)['sales'].shift(3)
    df_sorted['dy_lag1'] = df_sorted.groupby(entity_col)['d_sales'].shift(1)
    
    # Difference equations
    df_diff = df_sorted.dropna(subset=['d_sales', 'd_sales_lag', 'd_advertising', 
                                        'y_lag2', 'y_lag3']).copy()
    
    X_diff = df_diff[['d_sales_lag', 'd_advertising']].values
    y_diff = df_diff['d_sales'].values
    Z_diff = df_diff[['y_lag2', 'y_lag3', 'd_advertising']].values
    
    # Level equations
    df_level = df_sorted.dropna(subset=['sales', 'sales_lag', 'advertising', 
                                         'dy_lag1']).copy()
    
    X_level = df_level[['sales_lag', 'advertising']].values
    y_level = df_level['sales'].values
    Z_level = df_level[['dy_lag1', 'advertising']].values
    
    # Stack equations
    X = np.vstack([X_diff, X_level])
    y = np.concatenate([y_diff, y_level])
    Z = np.vstack([Z_diff, Z_level])
    
    # GMM estimation (as before)
    ZtZ_inv = np.linalg.inv(Z.T @ Z)
    PZ = Z @ ZtZ_inv @ Z.T
    
    XtPZX = X.T @ PZ @ X
    XtPZy = X.T @ PZ @ y
    
    beta = np.linalg.solve(XtPZX, XtPZy)
    
    residuals = y - X @ beta
    sigma2 = np.sum(residuals**2) / (len(residuals) - X.shape[1])
    var_beta = sigma2 * np.linalg.inv(XtPZX)
    se = np.sqrt(np.diag(var_beta))
    
    # J-test
    n = len(residuals)
    moments = (Z.T @ residuals) / n
    j_stat = n * moments.T @ ZtZ_inv @ moments
    j_df = Z.shape[1] - X.shape[1]
    j_pvalue = 1 - stats.chi2.cdf(j_stat, j_df)
    
    return {
        'rho': beta[0],
        'beta': beta[1],
        'se_rho': se[0],
        'se_beta': se[1],
        'n_obs': n,
        'n_instruments': Z.shape[1],
        'j_statistic': j_stat,
        'j_df': j_df,
        'j_pvalue': j_pvalue
    }

sys_gmm_results = system_gmm_simplified(df)

print("\nSystem GMM (Blundell-Bond) Results")
print("="*70)
print(f"{'Parameter':<20} {'True':>10} {'Estimate':>12} {'Std Error':>12} {'t-stat':>10}")
print("="*70)

rho_t_sys = sys_gmm_results['rho'] / sys_gmm_results['se_rho']
beta_t_sys = sys_gmm_results['beta'] / sys_gmm_results['se_beta']

print(f"{'ρ (persistence)':<20} {0.70:>10.4f} {sys_gmm_results['rho']:>12.4f} {sys_gmm_results['se_rho']:>12.4f} {rho_t_sys:>10.3f}")
print(f"{'β (advertising)':<20} {1.20:>10.4f} {sys_gmm_results['beta']:>12.4f} {sys_gmm_results['se_beta']:>12.4f} {beta_t_sys:>10.3f}")

print(f"\nModel Diagnostics:")
print(f"  Observations: {sys_gmm_results['n_obs']}")
print(f"  Instruments: {sys_gmm_results['n_instruments']}")
print(f"\nHansen J-test:")
print(f"  J-statistic: {sys_gmm_results['j_statistic']:.4f}")
print(f"  P-value: {sys_gmm_results['j_pvalue']:.4f}")

print("\n💡 System GMM typically more efficient than Difference GMM")
print("   (smaller standard errors when persistence is high)")

## 5. Compare All Estimators

In [ ]:
# Summary comparison
comparison = pd.DataFrame([
    {
        'Estimator': 'True Value',
        'ρ': 0.70,
        'SE(ρ)': np.nan,
        'β': 1.20,
        'SE(β)': np.nan
    },
    {
        'Estimator': 'Difference GMM',
        'ρ': gmm_results['rho'],
        'SE(ρ)': gmm_results['se_rho'],
        'β': gmm_results['beta'],
        'SE(β)': gmm_results['se_beta']
    },
    {
        'Estimator': 'System GMM',
        'ρ': sys_gmm_results['rho'],
        'SE(ρ)': sys_gmm_results['se_rho'],
        'β': sys_gmm_results['beta'],
        'SE(β)': sys_gmm_results['se_beta']
    }
])

print("\nEstimator Comparison:")
print("="*70)
print(comparison.to_string(index=False, float_format=lambda x: f"{x:.4f}" if not np.isnan(x) else ""))
print("\nNote: System GMM typically has smaller SEs (more efficient)")

## 6. Practical Considerations

### Instrument Proliferation

**Problem:** Number of instruments can exceed number of entities

**Consequences:**
- Overfitting
- Weak Hansen test
- Finite sample bias

**Solutions:**
- Limit instrument lags (e.g., use only t-2 and t-3)
- Collapse instruments (combine into smaller set)
- Use difference-in-Sargan test for subsets

In [ ]:
def count_instruments(t, max_lags=None):
    """
    Count number of instruments in Arellano-Bond GMM.
    
    For period t, can use lags 2, 3, ..., t-1
    Total: sum from t=3 to T of (t-2)
    """
    if max_lags is None:
        # All lags
        return sum(range(1, t-1))
    else:
        # Limit lags
        return min(max_lags, t-2) * (t - 2)

# Demonstrate instrument growth
t_values = [5, 10, 15, 20]
print("\nInstrument Count Growth:")
print("="*70)
print(f"{'T (periods)':<15} {'All lags':>15} {'Max 3 lags':>15} {'Max 2 lags':>15}")
print("="*70)

for t in t_values:
    all_lags = count_instruments(t)
    max3 = count_instruments(t, max_lags=3)
    max2 = count_instruments(t, max_lags=2)
    print(f"{t:<15} {all_lags:>15} {max3:>15} {max2:>15}")

print("\n⚠️  Recommendation: Limit instruments to avoid overfitting")
print("   Rule of thumb: # instruments < # entities")

---

## Exercises

### Exercise 1: Difference-in-Sargan Test

**Task:** Test validity of System GMM level moment conditions.

In [ ]:
# YOUR CODE HERE

diff_sargan_result = None

### Exercise 2: Robust Standard Errors

**Task:** Implement Windmeijer finite-sample correction for GMM SEs.

In [ ]:
# YOUR CODE HERE

windmeijer_ses = None

### Exercise 3: Simulate Weak Instruments

**Task:** Show that high persistence causes weak instrument problem in Difference GMM.

In [ ]:
# YOUR CODE HERE

weak_instrument_simulation = None

---

## Summary

### Key Takeaways

1. **GMM framework:** Efficient use of all moment conditions

2. **Arellano-Bond (Difference GMM):**
   - First difference to eliminate fixed effects
   - Use lagged levels as instruments
   - Weak instruments when $\rho$ high

3. **Blundell-Bond (System GMM):**
   - Adds level equations with lagged differences as instruments
   - More efficient, especially for persistent series
   - Requires stationarity assumption

4. **Diagnostics:**
   - Hansen J-test: Overidentifying restrictions
   - AR tests: Serial correlation
   - Instrument count: Avoid proliferation

5. **Best practices:**
   - Limit instrument lags
   - Use robust/clustered SEs
   - Always report diagnostics
   - Prefer System GMM for highly persistent series

### Estimator Selection Guide

| Persistence | T small | T large |
|-------------|---------|----------|
| Low (ρ < 0.5) | Diff GMM | FE adequate |
| Medium (0.5-0.8) | System GMM | Diff GMM or FE |
| High (ρ > 0.8) | System GMM | System GMM |

### When to Use GMM?

✅ **Use GMM when:**
- Lagged dependent variable present
- Small T (T < 20)
- Persistent series
- Endogenous regressors

❌ **Alternatives when:**
- Large T → Simple FE adequate
- No dynamics → Standard FE/RE
- Weak instruments → LIML or other methods

### Software

**Python:**
- `linearmodels.panel` (basic GMM)
- `statsmodels.GMM` (custom moments)

**Stata:**
- `xtabond2` (gold standard, most features)

**R:**
- `plm::pgmm()`
- `panelr` package

---

## Course Complete!

You now have a comprehensive understanding of panel regression methods:

1. Panel data structure and exploration
2. Fixed and random effects models
3. Model selection (Hausman test)
4. Diagnostics and robust inference
5. Dynamic panels and GMM estimation

**Next steps:**
- Apply to real data
- Read Arellano (2003) *Panel Data Econometrics*
- Explore extensions: nonlinear panels, unbalanced panels, large N large T asymptotics

---

**END OF MODULE 5**